<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ резюме из HeadHunter
   

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ModuleNotFoundError: No module named 'pandas'

# Исследование структуры данных

1. Прочитайте данные с помощью библиотеки Pandas. Совет: перед чтением обратите внимание на разделитель внутри файла. 

In [1]:
hh_data = pd.read_csv('data\dst-3.0_16_1_hh_database.csv', sep=";")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Григорий\AppData\Local\Temp\ipykernel_14652\3765413347.py:1: SyntaxWarning: invalid escape sequence '\d'
  hh_data = pd.read_csv('data\dst-3.0_16_1_hh_database.csv', sep=";")
C:\Users\Григорий\AppData\Local\Temp\ipykernel_14652\3765413347.py:1: SyntaxWarning: invalid escape sequence '\d'
  hh_data = pd.read_csv('data\dst-3.0_16_1_hh_database.csv', sep=";")


NameError: name 'pd' is not defined

2. Выведите несколько первых (последних) строк таблицы, чтобы убедиться в том, что ваши данные не повреждены. Ознакомьтесь с признаками и их структурой.

In [ ]:
hh_data.head()

3. Выведите основную информацию о числе непустых значений в столбцах и их типах в таблице.

4. Обратите внимание на информацию о числе непустых значений.

In [ ]:
hh_data.info()

5. Выведите основную статистическую информацию о столбцах.


In [ ]:
hh_data['Опыт работы'].nunique()
hh_data['Ищет работу на должность:'].mode()

# Преобразование данных

1. Начнем с простого - с признака **"Образование и ВУЗ"**. Его текущий формат это: **<Уровень образования год выпуска ВУЗ специальность...>**. Например:
* Высшее образование 2016 Московский авиационный институт (национальный исследовательский университет)...
* Неоконченное высшее образование 2000  Балтийская государственная академия рыбопромыслового флота…
Нас будет интересовать только уровень образования.

Создайте с помощью функции-преобразования новый признак **"Образование"**, который должен иметь 4 категории: "высшее", "неоконченное высшее", "среднее специальное" и "среднее".

Выполните преобразование, ответьте на контрольные вопросы и удалите признак "Образование и ВУЗ".

Совет: обратите внимание на структуру текста в столбце **"Образование и ВУЗ"**. Гарантируется, что текущий уровень образования соискателя всегда находится в первых 2ух слов и начинается с заглавной буквы. Воспользуйтесь этим.

*Совет: проверяйте полученные категории, например, с помощью метода unique()*


In [ ]:
def education_level(x):
    """Функция для приведения образования типу данных с четкими критериями"""
    x = x.split()
    if x[0] == 'Неоконченное':
        x = 'неоконченное высшее'
    elif x[0] == 'Высшее':
        x = 'высшее'
    else:
        if x[0] == 'Среднее' and x[1] == 'специальное':
            x = 'среднее специальное'
        else:
            x = 'среднее'
    return x
            
            
hh_data['Образование'] = hh_data['Образование и ВУЗ'].apply(education_level)
hh_data = hh_data.drop(columns='Образование и ВУЗ')

2. Теперь нас интересует столбец **"Пол, возраст"**. Сейчас он представлен в формате **<Пол , возраст , дата рождения >**. Например:
* Мужчина , 39 лет , родился 27 ноября 1979 
* Женщина , 21 год , родилась 13 января 2000
Как вы понимаете, нам необходимо выделить каждый параметр в отдельный столбец.

Создайте два новых признака **"Пол"** и **"Возраст"**. При этом важно учесть:
* Признак пола должен иметь 2 уникальных строковых значения: 'М' - мужчина, 'Ж' - женщина. 
* Признак возраста должен быть представлен целыми числами.

Выполните преобразование, ответьте на контрольные вопросы и удалите признак **"Пол, возраст"** из таблицы.

*Совет: обратите внимание на структуру текста в столбце, в части на то, как разделены параметры пола, возраста и даты рождения между собой - символом ' , '. 
Гарантируется, что структура одинакова для всех строк в таблице. Вы можете воспользоваться этим.*


In [ ]:
hh_data['Возраст'] = hh_data['Пол, возраст'].apply(lambda x: (x.split())[2])
hh_data['Возраст'] = hh_data['Возраст'].astype(int)

hh_data['Пол'] = hh_data['Пол, возраст'].apply(lambda x: 'М' if (x.split())[0] == 'Мужчина' else 'Ж')

hh_data = hh_data.drop(columns='Пол, возраст')

3. Следующим этапом преобразуем признак **"Опыт работы"**. Его текущий формат - это: **<Опыт работы: n лет m месяцев, периоды работы в различных компаниях…>**. 

Из столбца нам необходимо выделить общий опыт работы соискателя в месяцах, новый признак назовем "Опыт работы (месяц)"

Для начала обсудим условия решения задачи:
* Во-первых, в данном признаке есть пропуски. Условимся, что если мы встречаем пропуск, оставляем его как есть (функция-преобразование возвращает NaN)
* Во-вторых, в данном признаке есть скрытые пропуски. Для некоторых соискателей в столбце стоит значения "Не указано". Их тоже обозначим как NaN (функция-преобразование возвращает NaN)
* В-третьих, нас не интересует информация, которая описывается после указания опыта работы (периоды работы в различных компаниях)
* В-четвертых, у нас есть проблема: опыт работы может быть представлен только в годах или только месяцах. Например, можно встретить следующие варианты:
    * Опыт работы 3 года 2 месяца…
    * Опыт работы 4 года…
    * Опыт работы 11 месяцев…
    * Учитывайте эту особенность в вашем коде

Учитывайте эту особенность в вашем коде

В результате преобразования у вас должен получиться столбец, содержащий информацию о том, сколько месяцев проработал соискатель.
Выполните преобразование, ответьте на контрольные вопросы и удалите столбец **"Опыт работы"** из таблицы.


In [ ]:
def get_experience(arg):
    """Функция для перевода опыта работы в месяцы"""
    if arg is np.nan:
        return arg
    elif arg == 'Не указано':
        arg = np.nan
        return arg
    else:
        month_key_words = ['месяц', 'месяцев', 'месяца']
        year_key_words = ['год', 'лет', 'года']
        args_splited = arg.split(' ')
        month = 0
        year = 0
        for i in range(7):
            if args_splited[i] in month_key_words:
                month = args_splited[i-1]
                pass
            if args_splited[i] in year_key_words:
                year = args_splited[i-1]
                pass
        return int(year)*12 + int(month)
    
    
hh_data['Опыт работы(месяц)'] = hh_data['Опыт работы'].apply(get_experience)
hh_data= hh_data.drop(columns='Опыт работы')

4. Хорошо идем! Следующий на очереди признак "Город, переезд, командировки". Информация в нем представлена в следующем виде: **<Город , (метро) , готовность к переезду (города для переезда) , готовность к командировкам>**. В скобках указаны необязательные параметры строки. Например, можно встретить следующие варианты:

* Москва , не готов к переезду , готов к командировкам
* Москва , м. Беломорская , не готов к переезду, не готов к командировкам
* Воронеж , готов к переезду (Сочи, Москва, Санкт-Петербург) , готов к командировкам

Создадим отдельные признаки **"Город"**, **"Готовность к переезду"**, **"Готовность к командировкам"**. При этом важно учесть:

* Признак **"Город"** должен содержать только 4 категории: "Москва", "Санкт-Петербург" и "город-миллионник" (их список ниже), остальные обозначьте как "другие".

    Список городов-миллионников:
    
   <code>million_cities = ['Новосибирск', 'Екатеринбург','Нижний Новгород','Казань', 'Челябинск','Омск', 'Самара', 'Ростов-на-Дону', 'Уфа', 'Красноярск', 'Пермь', 'Воронеж','Волгоград']
    </code>
    Инфорация о метро, рядом с которым проживает соискатель нас не интересует.
* Признак **"Готовность к переезду"** должен иметь два возможных варианта: True или False. Обратите внимание, что возможны несколько вариантов описания готовности к переезду в признаке "Город, переезд, командировки". Например:
    * … , готов к переезду , …
    * … , не готова к переезду , …
    * … , готова к переезду (Москва, Санкт-Петербург, Ростов-на-Дону)
    * … , хочу переехать (США) , …
    
    Нас интересует только сам факт возможности или желания переезда.
* Признак **"Готовность к командировкам"** должен иметь два возможных варианта: True или False. Обратите внимание, что возможны несколько вариантов описания готовности к командировкам в признаке "Город, переезд, командировки". Например:
    * … , готов к командировкам , … 
    * … , готова к редким командировкам , …
    * … , не готов к командировкам , …
    
    Нас интересует только сам факт готовности к командировке.
    
    Еще один важный факт: при выгрузки данных у некоторых соискателей "потерялась" информация о готовности к командировкам. Давайте по умолчанию будем считать, что такие соискатели не готовы к командировкам.
    
Выполните преобразования и удалите столбец **"Город, переезд, командировки"** из таблицы.

*Совет: обратите внимание на то, что структура текста может меняться в зависимости от указания ближайшего метро. Учите это, если будете использовать порядок слов в своей программе.*


In [ ]:
#ваш код здесь
def get_city(x):
    """Функция для приведения признака к 4 нужным категориям"""
    x = x.split(' , ')
    city = ['Москва', 'Санкт-Петербург']
    million_cities = [
        'Новосибирск', 'Екатеринбург', 'Нижний Новгород', 'Казань', 'Челябинск',
        'Омск', 'Самара', 'Ростов-на-Дону', 'Уфа', 'Красноярск', 'Пермь',
        'Воронеж', 'Волгоград' 
        ]
    
    if x[0] in city:
        x = x[0]
    elif x[0] in million_cities:
        x = 'город-миллионник'
    else:
        x = 'другие'
    return x


def get_ready_to_move(arg):
    """Функция для определния готовности к переезду"""
    if ('не готов к переезду' in arg) or ('не готова к переезду' in arg):
        return False
    elif 'хочу' in arg:
        return True
    else:
        return True
    
    
def get_ready_for_bisiness_trips(arg):
    """Функция для определения готовности к командировкам"""
    if ('командировка' in arg):
        if ('не готов к командировкам' in arg) or('не готова к командировкам' in arg):
            return False
        else: 
            
            return True
    else:
        return False
    
hh_data['Город'] = hh_data['Город, переезд, командировки'].apply(get_city)
hh_data['Готовность к переезду']  = hh_data['Город, переезд, командировки'].apply(get_ready_to_move)
hh_data['Готовность к командировкам'] = hh_data['Город, переезд, командировки'].apply(get_ready_for_bisiness_trips)
hh_data= hh_data.drop(columns='Город, переезд, командировки')


In [ ]:
#заметка
#были написаны две собственные функции для готовности к переезду и командировкам. 
#Ошибку выявил уже после того как подглядел верную функцию в ответе.
#Понял, что проблема заключалась в разделение, не всегда пробел был с двух сторон от запятой.
def get_removal(x):
    x = x.split(' , ')
    x = x[-2].split()
    yes = ['готов', 'хочу']
    no = ['не']
    if x[0] in yes:
        return True
    else:
        return False


def get_btrip(x):
    x = x.split(' , ')
    x = x[-1].split()
    if x[0] == 'не':
        return False
    else:
        return True

5. Рассмотрим поближе признаки **"Занятость"** и **"График"**. Сейчас признаки представляют собой набор категорий желаемой занятости (полная занятость, частичная занятость, проектная работа, волонтерство, стажировка) и желаемого графика работы (полный день, сменный график, гибкий график, удаленная работа, вахтовый метод).
На сайте hh.ru соискатель может указывать различные комбинации данных категорий, например:
* полная занятость, частичная занятость
* частичная занятость, проектная работа, волонтерство
* полный день, удаленная работа
* вахтовый метод, гибкий график, удаленная работа, полная занятость

Такой вариант признаков имеет множество различных комбинаций, а значит множество уникальных значений, что мешает анализу. Нужно это исправить!

Давайте создадим признаки-мигалки для каждой категории: если категория присутствует в списке желаемых соискателем, то в столбце на месте строки рассматриваемого соискателя ставится True, иначе - False.

Такой метод преобразования категориальных признаков называется One Hot Encoding и его схема представлена на рисунке ниже:
<img src=https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/ohe.jpg>
Выполните данное преобразование для признаков "Занятость" и "График", ответьте на контрольные вопросы, после чего удалите их из таблицы

In [ ]:

key_word= ['полная занятость', 'частичная занятость', 'проектная работа', 'волонтерство', 'стажировка']
for word in key_word:
    hh_data[word] = hh_data['Занятость'].apply(lambda x: word in x)
    
key_word2= ['полный день', 'сменный график', 'гибкий график', 'удалённая работа', 'вахтовый метод']
for word in key_word2:
    hh_data[word] = hh_data['График'].apply(lambda x: word in x)
    
hh_data= hh_data.drop(columns='График')
hh_data= hh_data.drop(columns='Занятость')


6. (2 балла) Наконец, мы добрались до самого главного и самого важного - признака заработной платы **"ЗП"**. 
В чем наша беда? В том, что помимо желаемой заработной платы соискатель указывает валюту, в которой он бы хотел ее получать, например:
* 30000 руб.
* 50000 грн.
* 550 USD

Нам бы хотелось видеть заработную плату в единой валюте, например, в рублях. Возникает вопрос, а где взять курс валют по отношению к рублю?

На самом деле язык Python имеет в арсенале огромное количество возможностей получения данной информации, от обращения к API Центробанка, до использования специальных библиотек, например pycbrf. Однако, это не тема нашего проекта.

Поэтому мы пойдем в лоб: обратимся к специальным интернет-ресурсам для получения данных о курсе в виде текстовых файлов. Например, MDF.RU, данный ресурс позволяет удобно экспортировать данные о курсах различных валют и акций за указанные периоды в виде csv файлов. Мы уже сделали выгрузку курсов валют, которые встречаются в наших данных за период с 29.12.2017 по 05.12.2019. Скачать ее вы можете **на платформе**

Создайте новый DataFrame из полученного файла. В полученной таблице нас будут интересовать столбцы:
* "currency" - наименование валюты в ISO кодировке,
* "date" - дата, 
* "proportion" - пропорция, 
* "close" - цена закрытия (последний зафиксированный курс валюты на указанный день).


Перед вами таблица соответствия наименований иностранных валют в наших данных и их общепринятых сокращений, которые представлены в нашем файле с курсами валют. Пропорция - это число, за сколько единиц валюты указан курс в таблице с курсами. Например, для казахстанского тенге курс на 20.08.2019 составляет 17.197 руб. за 100 тенге, тогда итоговый курс равен - 17.197 / 100 = 0.17197 руб за 1 тенге.
Воспользуйтесь этой информацией в ваших преобразованиях.

<img src=https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/table.jpg>


Осталось только понять, откуда брать дату, по которой определяется курс? А вот же она - в признаке **"Обновление резюме"**, в нем содержится дата и время, когда соискатель выложил текущий вариант своего резюме. Нас интересует только дата, по ней бы и будем сопоставлять курсы валют.

Теперь у нас есть вся необходимая информация для того, чтобы создать признак "ЗП (руб)" - заработная плата в рублях.

После ответа на контрольные вопросы удалите исходный столбец заработной платы "ЗП" и все промежуточные столбцы, если вы их создавали.

Итак, давайте обсудим возможный алгоритм преобразования: 
1. Перевести признак "Обновление резюме" из таблицы с резюме в формат datetime и достать из него дату. В тот же формат привести признак "date" из таблицы с валютами.
2. Выделить из столбца "ЗП" сумму желаемой заработной платы и наименование валюты, в которой она исчисляется. Наименование валюты перевести в стандарт ISO согласно с таблицей выше.
3. Присоединить к таблице с резюме таблицу с курсами по столбцам с датой и названием валюты (подумайте, какой тип объединения надо выбрать, чтобы в таблице с резюме сохранились данные о заработной плате, изначально представленной в рублях). Значение close для рубля заполнить единицей 1 (курс рубля самого к себе)
4. Умножить сумму желаемой заработной платы на присоединенный курс валюты (close) и разделить на пропорцию (обратите внимание на пропуски после объединения в этих столбцах), результат занести в новый столбец "ЗП (руб)".


In [ ]:
exchange_rate = pd.read_csv('data\ExchangeRates.csv', sep=',')
exchange_rate = exchange_rate.drop(columns=['per', 'time', 'vol'])

#переводим время в дататайм, парарллельно создаем столбец с названием как в нашей таблице
exchange_rate['Обновление резюме'] = pd.to_datetime(exchange_rate['date'], dayfirst=True).dt.date
#сразу вычисляем пропорцию
exchange_rate['proportion_rate'] = exchange_rate['close']/exchange_rate['proportion']
exchange_rate = exchange_rate.drop(columns=['close', 'proportion'])

#переменовываем столбец для будущего объединения
exchange_rate.rename(columns={'currency': 'Валюта'}, inplace=True)


def get_iso(x):
    """Функция для преобразования валюты в ISO"""
    dict_currency = {'грн.': 'UAH', 'USD': 'USD', 'EUR': 'EUR', 'бел.руб.': 'BYN', 'KGS': 'KGS', 'сум': 'UZS', 'AZN': 'AZN', 'KZT': 'KZT'}
    if x == 'руб.':
        return x
    else:
        x= dict_currency[x]
        return x


def get_valuta(x):
    """Функция для получаения аименование валюты в данных"""
    x= x.split()
    x= x[-1]
    return x


def get_zp(x):
    """Функция для получения числа из столбца ЗП"""
    x= x.split()
    x= float(x[0])
    return x


hh_data['Валюта'] = hh_data['ЗП'].apply(get_valuta)
hh_data['ЗП'] = hh_data['ЗП'].apply(get_zp)
hh_data['Валюта'] = hh_data['Валюта'].apply(get_iso)
hh_data['Обновление резюме'] = pd.to_datetime(hh_data['Обновление резюме'], dayfirst=True).dt.date
#объединение таблиц с помощью Merge
hh_data = pd.merge(hh_data, exchange_rate, how='left', on=['Валюта', 'Обновление резюме'])
#Заполняем пустые значения
hh_data['proportion_rate'] = hh_data['proportion_rate'].fillna(1)
hh_data['ЗП(руб)'] = hh_data['ЗП'] * hh_data['proportion_rate']

hh_data= hh_data.drop(columns=['ЗП', 'Валюта', 'date', 'proportion_rate'])


# Исследование зависимостей в данных

1. Постройте распределение признака **"Возраст"**. Опишите распределение, отвечая на следующие вопросы: чему равна мода распределения, каковы предельные значения признака, в каком примерном интервале находится возраст большинства соискателей? Есть ли аномалии для признака возраста, какие значения вы бы причислили к их числу?
*Совет: постройте гистограмму и коробчатую диаграмму рядом.*

In [ ]:
fig1 = px.histogram(
    data_frame=hh_data,
    x='Возраст',
    title='Возрастное распределение'
)
fig1.show()

fig2 = px.box(
    data_frame=hh_data,
    x='Возраст',
    title='Возрастное распределение'
)
fig2.show()

Мода равна 30 годам. 
График показывает, что возрастной диапазон соискателей находится в интервале от 14 до 100 лет.
Большинство соискателей находится в диапазоне от 27 до 36 лет.
Возрастная аномалия в 100 лет. Вероятно, существует человек с таким возрастом и возможно он может еще заниматься какие то делом. Но в нашем случае данное отклонение можно приравнять к аномалии.

2. Постройте распределение признака **"Опыт работы (месяц)"**. Опишите данное распределение, отвечая на следующие вопросы: чему равна мода распределения, каковы предельные значения признака, в каком примерном интервале находится опыт работы большинства соискателей? Есть ли аномалии для признака опыта работы, какие значения вы бы причислили к их числу?
*Совет: постройте гистограмму и коробчатую диаграмму рядом.*

In [ ]:
fig3 = px.box(
    data_frame=hh_data,
    x='Опыт работы(месяц)',
    title='Опыт работы(месяц)'
)
fig3.show()

fig1 = px.histogram(
    data_frame=hh_data,
    x='Возраст',
    title='Возрастное распределение'
)
fig1.show()

Мода распределения находится в диапозоне 80-84 месяцев опыта работы. 

от 5 до 180 месяцев - в таком интервале находится опыт большинства соискателей. Минимальное значение 0 месяцев, максимальное 1188 месяцев.

Имеется аномалия в 1188 месяцев. 

3. Постройте распределение признака **"ЗП (руб)"**. Опишите данное распределение, отвечая на следующие вопросы: каковы предельные значения признака, в каком примерном интервале находится заработная плата большинства соискателей? Есть ли аномалии для признака возраста? Обратите внимание на гигантские размеры желаемой заработной платы.
*Совет: постройте гистограмму и коробчатую диаграмму рядом.*


In [ ]:
fig5 = px.box(
    data_frame=hh_data,
    x='ЗП(руб)',
    title='Заработная плата'
)
fig5.show()

fig5 = px.histogram(
    data_frame=hh_data,
    x='ЗП(руб)',
    title='Заработная плата'
)
fig5.show()

Мода равна 50 тысячам рублей.
Минимальное значение прихнака 0, максимальное более 24 миллионов. от 25 до 60 тысяч рублей находится ожидаемая заработная плана соискателей.
Значения более 1 миллиона рублей причилил бы к аномалиям.

4. Постройте диаграмму, которая показывает зависимость **медианной** желаемой заработной платы (**"ЗП (руб)"**) от уровня образования (**"Образование"**). Используйте для диаграммы данные о резюме, где желаемая заработная плата меньше 1 млн рублей.
*Сделайте выводы по представленной диаграмме: для каких уровней образования наблюдаются наибольшие и наименьшие уровни желаемой заработной платы? Как вы считаете, важен ли признак уровня образования при прогнозировании заработной платы?*

In [ ]:
data_group = hh_data[hh_data['ЗП(руб)']<1000000].groupby(by='Образование', as_index=False)['ЗП(руб)'].median()

fig7 = px.histogram(
    data_frame=data_group,
    x='Образование',
    y='ЗП(руб)',
    title='Зависимость средней ЗП от уровня образования'
)
fig7.show()

Наивысшие ожидание от зарабатной платы у соискатель с высшим образование - 60 тысяч рублей.

Соискатели с средним и среднем специальным образование имеют одинаковый показатель по ожидаемой зарабатной плате - 40 тысяч рублей.

Признак образование при прогнозирование заработной платы важен.

5. Постройте диаграмму, которая показывает распределение желаемой заработной платы (**"ЗП (руб)"**) в зависимости от города (**"Город"**). Используйте для диаграммы данные о резюме, где желая заработная плата меньше 1 млн рублей.
*Сделайте выводы по полученной диаграмме: как соотносятся медианные уровни желаемой заработной платы и их размах в городах? Как вы считаете, важен ли признак города при прогнозировании заработной платы?*

In [ ]:
fig= px.box(
    data_frame=hh_data[hh_data['ЗП(руб)']<1000000],
    x='ЗП(руб)',
    y='Город',
)
fig.show()

Медианный уровень заработной платы выше всего в столице. Большая часть соискателей указывает желаемую заработную плату в районе от 60 до 150 тысяч. Медианное значение находится ближе к левому краю и составляет 85 тысяч. Размах средней заработной платы в Москве сильно "растянут".
Соискатели в городах миллиониках и других более "точно" описывают свои желания в заработной плате, их среднее значение не так растянуто как в Москве, хотя так же наблюдается сдвиг медианного значения к левому краю.
Признак города при прогнозирование заработной платы важен.

6. Постройте **многоуровневую столбчатую диаграмму**, которая показывает зависимость медианной заработной платы (**"ЗП (руб)"**) от признаков **"Готовность к переезду"** и **"Готовность к командировкам"**. Проанализируйте график, сравнив уровень заработной платы в категориях.

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=hh_data,
    x='Готовность к переезду',
    y='ЗП(руб)',
    hue='Готовность к командировкам',
    palette='Blues'
)
plt.title('Медианная ЗП в зависимости от готовности к переезду и командировкам', fontsize=14, pad=20)
plt.xlabel('Готовность к переезду', fontsize=12)
plt.ylabel('Медианная ЗП (руб)', fontsize=12)


Соискатели готовые к переезду указывают заработную плату выше, чем соискатели не готовые менять своё место жительства. Так же отчетливо видно, что при отрительном отношение к командировкам, диаграмма показывает значение заработной платы ниже чем при согласии.

7. Постройте сводную таблицу, иллюстрирующую зависимость **медианной** желаемой заработной платы от возраста (**"Возраст"**) и образования (**"Образование"**). На полученной сводной таблице постройте **тепловую карту**. Проанализируйте тепловую карту, сравнив показатели внутри групп.

In [ ]:
hh_pivot_table = pd.pivot_table(
    hh_data, values='ЗП(руб)',
    columns='Возраст',
    index='Образование',
    aggfunc='median'
    )

heatmap = sns.heatmap(data=hh_pivot_table, cmap='YlGnBu')
heatmap.set_title('Тепловая карта заработной платы', fontsize=16);


Наиболее плавный и без резкого контраста график показывает в категории среднего специального образования. заработная плата растет медлено и не превышает значения более 60 тысяч рублей. Наблюдается резкий скачок в районе 67 лет, возможно это связано с аномалией в данных.

Сосискатели с высшим образование показывают более быстрый рост в ожидаемой заработной плате с по отношению к возрасту. Это может быть связано с желание постонно расти как финансово так и професиионально. Имеется высокий уровень ожидания ЗП в районе 16 лет, возможно это аномалия или отсутствия понимания как формируется рынок труда. Данные по высшему образованию охватывают бОльшую возрастную категорию, чем другие. Наибольшие ожидания от заработной платы находятся в диапозоне от 38 до 48 лет.

График неоконченного высшего и среднего образования схиодтся в поведении. Он не однородный с выбросами в разные промежутки. У неоконченного высшего наибольшие ожидания от зарплаты формируются у людей в возрасте от 44 до 50 лет. У среднего образования явного фомирования по возрасту не проглядывается.


8. Постройте **диаграмму рассеяния**, показывающую зависимость опыта работы (**"Опыт работы (месяц)"**) от возраста (**"Возраст"**). Опыт работы переведите из месяцев в года, чтобы признаки были в едином масштабе. Постройте на графике дополнительно прямую, проходящую через точки (0, 0) и (100, 100). Данная прямая соответствует значениям, когда опыт работы равен возрасту человека. Точки, лежащие на этой прямой и выше нее - аномалии в наших данных (опыт работы больше либо равен возрасту соискателя)

In [ ]:
hh_data['Опыт работы(год)'] = round((hh_data['Опыт работы(месяц)']/12), 2)

fig = px.scatter(
    data_frame=hh_data,
    x='Возраст',
    y='Опыт работы(год)'
)
#нижнюю часть функции честно подстмотрел и интернете. остальное сам. честно.
fig.add_shape(
    type='line',
    x0=0, y0=0,
    x1=100, y1=100,
    line=dict(color='black', width=2, dash='dash'),
    name='y = x (возраст = опыт)'
)
fig.show()

На диаграмме наблюдаются 7 выбросов явных, когда человек проработал больше чем прожил. Видно что рост опыта и возраста в основном плавный. Но так же отчетливо видно что нижняя граница не поднимается вместе с верхней. Значит на рынок труда приходят взрослые люди не имеющие достаточного опыта к своим годам.  Имеются некоторые "выбросы", которые не пересекли линию, но тоже являются аномалиями. 

**Дополнительные баллы**

Для получения 2 дополнительных баллов по разведывательному анализу постройте еще два любых содержательных графика или диаграммы, которые помогут проиллюстрировать влияние признаков/взаимосвязь между признаками/распределения признаков. Приведите выводы по ним. Желательно, чтобы в анализе участвовали признаки, которые мы создавали ранее в разделе "Преобразование данных".


In [ ]:
# ваш код здесь

ваши выводы здесь

# Очистка данных

1. Начнем с дубликатов в наших данных. Найдите **полные дубликаты** в таблице с резюме и удалите их. 

In [ ]:
dupl_columns = list(hh_data.columns)
mask = hh_data.duplicated(subset=dupl_columns)
hh_duplicates = hh_data[mask]
hh_data = hh_data.drop_duplicates(subset=dupl_columns)
print(f'Число найденных дубликатов: {hh_duplicates.shape[0]}')


2. Займемся пропусками. Выведите информацию **о числе пропусков** в столбцах. 

In [ ]:
cols_null_percent = hh_data.isnull().sum()
cols_with_null = cols_null_percent[cols_null_percent>0].sort_values(ascending=False)
display(cols_with_null)


3. Итак, у нас есть пропуски в 3ех столбцах: **"Опыт работы (месяц)"**, **"Последнее/нынешнее место работы"**, **"Последняя/нынешняя должность"**. Поступим следующим образом: удалите строки, где есть пропуск в столбцах с местом работы и должностью. Пропуски в столбце с опытом работы заполните **медианным** значением.

In [ ]:
hh_data = hh_data.dropna(axis=0, how='any', subset=['Последняя/нынешняя должность', 'Последнее/нынешнее место работы'])
values = {'Опыт работы(месяц)': hh_data['Опыт работы(месяц)'].median()}
hh_data = hh_data.fillna(values)

4. Мы добрались до ликвидации выбросов. Сначала очистим данные вручную. Удалите резюме, в которых указана заработная плата либо выше 1 млн. рублей, либо ниже 1 тыс. рублей.

In [ ]:
mask = (hh_data['ЗП(руб)'] > 1000000) | (hh_data['ЗП(руб)'] < 1000)
indices_to_drop = hh_data[mask].index
hh_data = hh_data.drop(indices_to_drop, axis=0)

5. В процессе разведывательного анализа мы обнаружили резюме, в которых **опыт работы в годах превышал возраст соискателя**. Найдите такие резюме и удалите их из данных


In [ ]:
mask = hh_data['Опыт работы(год)']>hh_data['Возраст']
indices_to_drop = hh_data[mask].index
hh_data = hh_data.drop(indices_to_drop, axis=0)

6. В результате анализа мы обнаружили потенциальные выбросы в признаке **"Возраст"**. Это оказались резюме людей чересчур преклонного возраста для поиска работы. Попробуйте построить распределение признака в **логарифмическом масштабе**. Добавьте к графику линии, отображающие **среднее и границы интервала метода трех сигм**. Напомним, сделать это можно с помощью метода axvline. Например, для построение линии среднего будет иметь вид:

`histplot.axvline(log_age.mean(), color='k', lw=2)`

В какую сторону асимметрично логарифмическое распределение? Напишите об этом в комментарии к графику.
Найдите выбросы с помощью метода z-отклонения и удалите их из данных, используйте логарифмический масштаб. Давайте сделаем послабление на **1 сигму** (возьмите 4 сигмы) в **правую сторону**.

Выведите таблицу с полученными выбросами и оцените, с каким возрастом соискатели попадают под категорию выбросов?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
log_age = np.log(hh_data['Возраст'] + 1)
histplot = sns.histplot(log_age, bins=30, ax=ax)
histplot.axvline(log_age.mean(), color='k', lw=2)
histplot.axvline(log_age.mean()+ 4 *log_age.std(), color='k', ls='--', lw=2)
histplot.axvline(log_age.mean()- 3 *log_age.std(), color='k', ls='--', lw=2)
histplot.set_title('Log Age Distribution');

def outliers_z_score_mod(data, feature, left=3, right=3, log_scale=False):
    if log_scale:
        x = np.log(data[feature]+1)
    else:
        x = data[feature]
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - left * sigma
    upper_bound = mu + right * sigma
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned
outliers, cleaned_data = outliers_z_score_mod(hh_data, 'Возраст', left=3,  right=4, log_scale=True)
print(outliers.shape[0])

На данном графике наблядаем правосторонюю асимметрию. Видны выброс с левой стороны границы.